# FlashRank Reranker Experiment for Turkish Legal RAG

This notebook evaluates the effect of adding a Cross-Encoder Reranker to the existing Turkish Legal RAG pipeline.

The notebook is independent from `03_rag_generation.ipynb`, but it uses the same project files, retrieval corpus, FAISS index, test set, prompt style, LLM, and manual evaluation setup.

The only changed component is the reranking step after hybrid retrieval.

In [1]:
!pip install -q -U flashrank sentence-transformers faiss-cpu rank-bm25 transformers accelerate bitsandbytes rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 75.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
project_path = "/content/drive/MyDrive/turkish_legal_rag"
processed_path = f"{project_path}/data/processed"
faiss_path = f"{project_path}/outputs/faiss"
metrics_path = f"{project_path}/outputs/metrics"

print("Project path:", project_path)
print("Processed path:", processed_path)
print("FAISS path:", faiss_path)
print("Metrics path:", metrics_path)

Project path: /content/drive/MyDrive/turkish_legal_rag
Processed path: /content/drive/MyDrive/turkish_legal_rag/data/processed
FAISS path: /content/drive/MyDrive/turkish_legal_rag/outputs/faiss
Metrics path: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics


In [4]:
import os
import re
import json
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import faiss

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

from flashrank import Ranker, RerankRequest

In [5]:
chunks_df = pd.read_csv(f"{processed_path}/retrieval_corpus.csv")
test_qa_df = pd.read_csv(f"{processed_path}/test_qa.csv")

print("Chunks shape:", chunks_df.shape)
print("Test QA shape:", test_qa_df.shape)

chunks_df.head()

Chunks shape: (3775, 5)
Test QA shape: (1500, 2)


,chunk_id,source_context_id,source,chunk_text,chunk_len
0,chunk_000000,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,Türk Vatanı ve Milletinin ebedi varlığını ve Y...,263
1,chunk_000001,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,Dünya milletleri ailesinin eşit haklara sahip ...,194
2,chunk_000002,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,"Millet iradesinin mutlak üstünlüğü, egemenliği...",276
3,chunk_000003,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,"Kuvvetler ayrımının, Devlet organları arasında...",256
4,chunk_000004,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,"Hiçbir faaliyetin Türk milli menfaatlerinin, T...",370


In [6]:
test_qa_df.head()

,question,answer
0,Anayasanın 90. Maddesi Nasıl Uygulanır?,Milletlerarası antlaşmaların TBMM tarafından o...
1,Hukukta 'legitimate expectation' nedir?,"Legitimate expectation, bir kişinin belirli bi..."
2,"Anayasa madde 172'ye göre, devletin sanayi ve ...","Anayasa madde 172'ye göre, devlet, sanayi ve t..."
3,"Anayasa madde 158, uyuşmazlık mahkemesi'nin ku...","Anayasa madde 158'e göre, uyuşmazlık mahkemesi..."
4,"Bir grup avukat, Türkiye Büyük Millet Meclisi ...","Anayasanın 94. Maddesi, Türkiye Büyük Millet M..."


In [7]:
test_eval_df = test_qa_df.sample(n=20, random_state=42).reset_index(drop=True)

print(test_eval_df.shape)
test_eval_df.head()

(20, 2)


,question,answer
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...


In [8]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. LLM loading may be very slow or may fail.")

CUDA available: True
GPU: Tesla T4


In [9]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [10]:
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(embedding_model_name)

index = faiss.read_index(f"{faiss_path}/baseline_faiss.index")

print("Embedding model loaded:", embedding_model_name)
print("Chunks:", chunks_df.shape)
print("FAISS vectors:", index.ntotal)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Chunks: (3775, 5)
FAISS vectors: 3775


In [11]:
def simple_turkish_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zçğıöşü0-9\s]", " ", text)
    tokens = text.split()

    stopwords = {
        "ve", "veya", "ile", "de", "da", "bir", "bu", "şu", "o",
        "için", "gibi", "olarak", "olan", "kadar", "ise", "ancak",
        "çok", "daha", "en", "mi", "mı", "mu", "mü"
    }

    return [t for t in tokens if t not in stopwords and len(t) > 1]

In [12]:
tokenized_corpus = [
    simple_turkish_tokenize(text)
    for text in chunks_df["chunk_text"].astype(str).tolist()
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 ready.")

BM25 ready.


In [13]:
def min_max_normalize(scores):
    scores = np.array(scores, dtype=np.float32)

    if scores.max() == scores.min():
        return np.zeros_like(scores)

    return (scores - scores.min()) / (scores.max() - scores.min())


def hybrid_retrieve_top_k(query, model, index, chunks_df, bm25, k=5, alpha=0.5):
    query_embedding = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_embedding)

    dense_scores, dense_indices = index.search(query_embedding, len(chunks_df))

    dense_scores = dense_scores[0]
    dense_indices = dense_indices[0]

    dense_score_map = {
        int(idx): float(score)
        for idx, score in zip(dense_indices, dense_scores)
    }

    dense_all_scores = np.array([
        dense_score_map.get(i, 0.0)
        for i in range(len(chunks_df))
    ])

    tokenized_query = simple_turkish_tokenize(query)
    bm25_scores = np.array(bm25.get_scores(tokenized_query))

    dense_norm = min_max_normalize(dense_all_scores)
    bm25_norm = min_max_normalize(bm25_scores)

    final_scores = alpha * dense_norm + (1 - alpha) * bm25_norm

    top_indices = np.argsort(final_scores)[::-1][:k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "source": chunks_df.iloc[idx]["source"],
            "score": float(final_scores[idx]),
            "dense_score": float(dense_norm[idx]),
            "bm25_score": float(bm25_norm[idx]),
            "chunk_text": chunks_df.iloc[idx]["chunk_text"]
        })

    return results

In [14]:
def detect_source_filter(query):
    q = str(query).lower()

    if "anayasa" in q or "anayasanın" in q:
        return "Türkiye Cumhuriyeti Anayasası"

    return None


def hybrid_retrieve_top_k_filtered(query, model, index, chunks_df, bm25, k=5, alpha=0.5):
    source_filter = detect_source_filter(query)

    candidate_df = chunks_df.copy()

    if source_filter is not None:
        candidate_df = candidate_df[
            candidate_df["source"].astype(str).str.lower() == source_filter.lower()
        ].reset_index(drop=True)

    if len(candidate_df) == 0:
        candidate_df = chunks_df.copy()

    candidate_texts = candidate_df["chunk_text"].astype(str).tolist()

    candidate_embeddings = embedding_model.encode(
        candidate_texts,
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")

    faiss.normalize_L2(candidate_embeddings)

    temp_index = faiss.IndexFlatIP(candidate_embeddings.shape[1])
    temp_index.add(candidate_embeddings)

    tokenized_candidate_corpus = [
        simple_turkish_tokenize(text)
        for text in candidate_texts
    ]

    temp_bm25 = BM25Okapi(tokenized_candidate_corpus)

    return hybrid_retrieve_top_k(
        query,
        model,
        temp_index,
        candidate_df,
        temp_bm25,
        k=k,
        alpha=alpha
    )

In [15]:
test_results = hybrid_retrieve_top_k_filtered(
    "Egemenlik kime aittir?",
    embedding_model,
    index,
    chunks_df,
    bm25,
    k=5,
    alpha=0.5
)

for r in test_results:
    print("=" * 80)
    print(r["rank"], r["chunk_id"], r["score"], r["source"])
    print(r["chunk_text"][:500])

1 chunk_000269 0.9583597183227539 Türkiye Cumhuriyeti Anayasası
Madde 6 – Egemenlik, kayıtsız şartsız Milletindir. Türk Milleti, egemenliğini, Anayasanın koyduğu esaslara göre, yetkili organları eliyle kullanır.
2 chunk_003519 0.8980596661567688 Türk Ceza Kanunu
DÖRDÜNCÜ KISIM
Millete ve Devlete Karşı Suçlar ve Son Hükümler ÜÇÜNCÜ BÖLÜM
Devletin Egemenlik Alametlerine ve Organlarının Saygınlığına Karşı Suçlar
3 chunk_002321 0.8044273853302002 Türk Medeni Kanunu
Madde 960- Ortaklık genel kurulunda rehinli pay senetlerini temsil etmek yetkisi, rehin 
alacaklısına değil, pay sahibine aittir.
II I. Yönetim ve ödeme
4 chunk_001360 0.744124710559845 Ceza Muhakemesi Kanunu
Madde 12 – (1) Davaya bakmak yetkisi, suçun işlendiği yer mahkemesine aittir.
5 chunk_000268 0.7346979975700378 Türkiye Cumhuriyeti Anayasası
Madde 5 – Devletin temel amaç ve görevleri, Türk milletinin bağımsızlığını ve bütünlüğünü, ülkenin bölünmezliğini, Cumhuriyeti ve demokrasiyi korumak, kişilerin ve toplumun refah, huz

## FlashRank Reranker

In [16]:
reranker_model_name = "ms-marco-MultiBERT-L-12"

ranker = Ranker(
    model_name=reranker_model_name,
    cache_dir="/content/flashrank_cache"
)

print("FlashRank reranker loaded:", reranker_model_name)

ms-marco-MultiBERT-L-12.zip: 100%|██████████| 98.7M/98.7M [00:00<00:00, 215MiB/s]


FlashRank reranker loaded: ms-marco-MultiBERT-L-12


In [17]:
def rerank_retrieved_chunks(question, retrieved_results, top_k=3):
    passages = []

    for i, item in enumerate(retrieved_results):
        passages.append({
            "id": i,
            "text": item["chunk_text"],
            "meta": {
                "chunk_id": item["chunk_id"],
                "source": item["source"],
                "original_rank": item["rank"],
                "hybrid_score": item["score"],
                "dense_score": item["dense_score"],
                "bm25_score": item["bm25_score"]
            }
        })

    rerank_request = RerankRequest(
        query=question,
        passages=passages
    )

    reranked = ranker.rerank(rerank_request)
    reranked = reranked[:top_k]

    final_results = []

    for new_rank, item in enumerate(reranked, start=1):
        original_index = int(item["id"])
        original_item = retrieved_results[original_index]

        final_results.append({
            "rank": new_rank,
            "chunk_id": original_item["chunk_id"],
            "source": original_item["source"],
            "rerank_score": float(item["score"]),
            "original_rank": original_item["rank"],
            "original_hybrid_score": original_item["score"],
            "dense_score": original_item["dense_score"],
            "bm25_score": original_item["bm25_score"],
            "chunk_text": original_item["chunk_text"]
        })

    return final_results

In [18]:
def hybrid_retrieve_with_reranker(
    query,
    model,
    index,
    chunks_df,
    bm25,
    candidate_k=10,
    final_k=3,
    alpha=0.5
):
    candidates = hybrid_retrieve_top_k_filtered(
        query,
        model,
        index,
        chunks_df,
        bm25,
        k=candidate_k,
        alpha=alpha
    )

    reranked_results = rerank_retrieved_chunks(
        question=query,
        retrieved_results=candidates,
        top_k=final_k
    )

    return reranked_results

In [19]:
sample_question = "Cumhurbaşkanının görev süresi kaç yıldır?"

base_candidates = hybrid_retrieve_top_k_filtered(
    sample_question,
    embedding_model,
    index,
    chunks_df,
    bm25,
    k=10,
    alpha=0.5
)

reranked_results = hybrid_retrieve_with_reranker(
    sample_question,
    embedding_model,
    index,
    chunks_df,
    bm25,
    candidate_k=10,
    final_k=3,
    alpha=0.5
)

print("QUESTION:")
print(sample_question)

print("\n" + "=" * 100)
print("ORIGINAL HYBRID TOP 10")
for r in base_candidates:
    print("-" * 80)
    print("Original rank:", r["rank"])
    print("Chunk ID:", r["chunk_id"])
    print("Hybrid score:", r["score"])
    print("Source:", r["source"])
    print(r["chunk_text"][:500])

print("\n" + "=" * 100)
print("RERANKED TOP 3")
for r in reranked_results:
    print("-" * 80)
    print("New rank:", r["rank"])
    print("Original rank:", r["original_rank"])
    print("Chunk ID:", r["chunk_id"])
    print("Rerank score:", r["rerank_score"])
    print("Original hybrid score:", r["original_hybrid_score"])
    print("Source:", r["source"])
    print(r["chunk_text"][:500])

QUESTION:
Cumhurbaşkanının görev süresi kaç yıldır?

ORIGINAL HYBRID TOP 10
--------------------------------------------------------------------------------
Original rank: 1
Chunk ID: chunk_000043
Hybrid score: 1.0
Source: Türkiye Cumhuriyeti Anayasası
Cumhurbaşkanı, kırk yaşını doldurmuş, yükseköğrenim yapmış, milletvekili seçilme yeterliliğine sahip Türk vatandaşları arasından, doğrudan halk tarafından seçilir. Cumhurbaşkanının görev süresi beş yıldır. Bir kimse en fazla iki defa Cumhurbaşkanı seçilebilir.
--------------------------------------------------------------------------------
Original rank: 2
Chunk ID: chunk_000086
Hybrid score: 0.9484720230102539
Source: Türkiye Cumhuriyeti Anayasası
Seçimlerinin birlikte yenilenmesine karar verilen Meclisin ve Cumhurbaşkanının yetki ve görevleri, yeni Meclisin ve Cumhurbaşkanının göreve başlamasına kadar devam eder. Bu şekilde seçilen Meclis ve Cumhurbaşkanının görev süreleri de beş yıldır. İ. Milli Savunma 1. Başkomutanlık ve Genelkurmay

## Conservative Hybrid + Reranker Fusion

The pure FlashRank reranker was first tested as a sanity check. However, in some cases, it moved the exact-answer chunk below less direct chunks.

Therefore, we use a conservative fusion strategy instead of relying only on the reranker score.

The final ranking combines:
- original hybrid retrieval rank
- FlashRank reranker rank

This prevents the reranker from completely overriding strong hybrid retrieval results.

In [20]:
def debug_full_rerank(question, candidate_k=10):
    candidates = hybrid_retrieve_top_k_filtered(
        question,
        embedding_model,
        index,
        chunks_df,
        bm25,
        k=candidate_k,
        alpha=0.5
    )

    reranked_all = rerank_retrieved_chunks(
        question=question,
        retrieved_results=candidates,
        top_k=candidate_k
    )

    rows = []

    for item in reranked_all:
        rows.append({
            "new_rank": item["rank"],
            "original_rank": item["original_rank"],
            "chunk_id": item["chunk_id"],
            "source": item["source"],
            "rerank_score": item["rerank_score"],
            "original_hybrid_score": item["original_hybrid_score"],
            "chunk_preview": item["chunk_text"][:250]
        })

    return pd.DataFrame(rows)

In [21]:
debug_full_rerank("Cumhurbaşkanının görev süresi kaç yıldır?", candidate_k=10)

,new_rank,original_rank,chunk_id,source,rerank_score,original_hybrid_score,chunk_preview
0,1,2,chunk_000086,Türkiye Cumhuriyeti Anayasası,0.999702,0.948472,Seçimlerinin birlikte yenilenmesine karar veri...
1,2,6,chunk_000061,Türkiye Cumhuriyeti Anayasası,0.999672,0.644201,Cumhurbaşkanının görevde bulunduğu sürede işle...
2,3,10,chunk_000065,Türkiye Cumhuriyeti Anayasası,0.999612,0.580269,Cumhurbaşkanının hastalık ve yurt dışına çıkma...
3,4,9,chunk_000057,Türkiye Cumhuriyeti Anayasası,0.999595,0.584117,"Cumhurbaşkanı, ayrıca Anayasada ve kanunlarda ..."
4,5,4,chunk_000025,Türkiye Cumhuriyeti Anayasası,0.999543,0.761215,Türkiye Büyük Millet Meclisi Başkanlık Divanı ...
5,6,3,chunk_000588,Bilgi Edinme Kanunu,0.999521,0.768831,Kurul üyelerinin görev süreleri dört yıldır. G...
6,7,7,chunk_000749,Ceza Muhakemesi Kanunu,0.999366,0.638479,(1) Ağır ceza mahkemesinin görevine girmeyen i...
7,8,5,chunk_000174,Türkiye Cumhuriyeti Anayasası,0.999350,0.657341,Anayasa Mahkemesi üyeleri arasından gizli oyla...
8,9,1,chunk_000043,Türkiye Cumhuriyeti Anayasası,0.998867,1.000000,"Cumhurbaşkanı, kırk yaşını doldurmuş, yükseköğ..."
9,10,8,chunk_000714,Ceza Muhakemesi Kanunu,0.998793,0.611406,Madde 102 – (1) (Değişik: 6/12/2006 – 5560/18 ...


In [22]:
def hybrid_retrieve_with_reranker_fusion(
    query,
    model,
    index,
    chunks_df,
    bm25,
    candidate_k=10,
    final_k=3,
    alpha=0.5,
    hybrid_weight=0.7,
    rerank_weight=0.3
):
    candidates = hybrid_retrieve_top_k_filtered(
        query,
        model,
        index,
        chunks_df,
        bm25,
        k=candidate_k,
        alpha=alpha
    )

    reranked_all = rerank_retrieved_chunks(
        question=query,
        retrieved_results=candidates,
        top_k=candidate_k
    )

    # Reranker rank map
    rerank_rank_map = {
        item["chunk_id"]: item["rank"]
        for item in reranked_all
    }

    fused_results = []

    for item in candidates:
        chunk_id = item["chunk_id"]

        original_rank = item["rank"]
        rerank_rank = rerank_rank_map.get(chunk_id, candidate_k + 1)

        # Rank-based scoring:
        # rank 1 -> 1.0
        # rank 2 -> 0.5
        # rank 3 -> 0.333...
        hybrid_rank_score = 1 / original_rank
        rerank_rank_score = 1 / rerank_rank

        fusion_score = (
            hybrid_weight * hybrid_rank_score
            + rerank_weight * rerank_rank_score
        )

        fused_results.append({
            "rank": None,
            "chunk_id": item["chunk_id"],
            "source": item["source"],
            "fusion_score": float(fusion_score),
            "original_rank": original_rank,
            "rerank_rank": rerank_rank,
            "original_hybrid_score": item["score"],
            "chunk_text": item["chunk_text"]
        })

    fused_results = sorted(
        fused_results,
        key=lambda x: x["fusion_score"],
        reverse=True
    )

    final_results = []

    for new_rank, item in enumerate(fused_results[:final_k], start=1):
        item["rank"] = new_rank
        final_results.append(item)

    return final_results

In [23]:
fusion_results = hybrid_retrieve_with_reranker_fusion(
    "Cumhurbaşkanının görev süresi kaç yıldır?",
    embedding_model,
    index,
    chunks_df,
    bm25,
    candidate_k=10,
    final_k=3,
    alpha=0.5,
    hybrid_weight=0.7,
    rerank_weight=0.3
)

for r in fusion_results:
    print("=" * 80)
    print("New rank:", r["rank"])
    print("Chunk ID:", r["chunk_id"])
    print("Original rank:", r["original_rank"])
    print("Rerank rank:", r["rerank_rank"])
    print("Fusion score:", r["fusion_score"])
    print("Source:", r["source"])
    print(r["chunk_text"][:500])

New rank: 1
Chunk ID: chunk_000043
Original rank: 1
Rerank rank: 9
Fusion score: 0.7333333333333333
Source: Türkiye Cumhuriyeti Anayasası
Cumhurbaşkanı, kırk yaşını doldurmuş, yükseköğrenim yapmış, milletvekili seçilme yeterliliğine sahip Türk vatandaşları arasından, doğrudan halk tarafından seçilir. Cumhurbaşkanının görev süresi beş yıldır. Bir kimse en fazla iki defa Cumhurbaşkanı seçilebilir.
New rank: 2
Chunk ID: chunk_000086
Original rank: 2
Rerank rank: 1
Fusion score: 0.6499999999999999
Source: Türkiye Cumhuriyeti Anayasası
Seçimlerinin birlikte yenilenmesine karar verilen Meclisin ve Cumhurbaşkanının yetki ve görevleri, yeni Meclisin ve Cumhurbaşkanının göreve başlamasına kadar devam eder. Bu şekilde seçilen Meclis ve Cumhurbaşkanının görev süreleri de beş yıldır. İ. Milli Savunma 1. Başkomutanlık ve Genelkurmay Başkanlığı
New rank: 3
Chunk ID: chunk_000588
Original rank: 3
Rerank rank: 6
Fusion score: 0.2833333333333333
Source: Bilgi Edinme Kanunu
Kurul üyelerinin görev sürele

In [24]:
CANDIDATE_K = 10
FINAL_CONTEXT_K = 3
ALPHA = 0.5

HYBRID_WEIGHT = 0.7
RERANK_WEIGHT = 0.3

print("Candidate K:", CANDIDATE_K)
print("Final context K:", FINAL_CONTEXT_K)
print("Alpha:", ALPHA)
print("Hybrid weight:", HYBRID_WEIGHT)
print("Rerank weight:", RERANK_WEIGHT)

Candidate K: 10
Final context K: 3
Alpha: 0.5
Hybrid weight: 0.7
Rerank weight: 0.3


In [25]:
def build_strict_rag_prompt(question, retrieved_contexts):
    context_text = "\n\n".join([
        f"[Context {i+1}]\n{ctx}"
        for i, ctx in enumerate(retrieved_contexts)
    ])

    prompt = f"""
Sen Türk hukuk metinleri için çalışan dikkatli bir soru-cevap asistanısın.

Kurallar:
- Cevabı SADECE verilen bağlama göre ver.
- Bağlamda açıkça yazmayan çıkarımları yapma.
- Birden fazla bağlam çelişirse en doğrudan cevap veren bağlamı kullan.
- Cevap Türkçe olmalı.
- Cevap kısa ve net olmalı.
- İngilizce açıklama, "Therefore", "Context 1" gibi ifadeler yazma.

Bağlam:
{context_text}

Soru:
{question}

Kısa cevap:
"""
    return prompt.strip()


def clean_generated_answer(text):
    text = str(text)

    if "Kısa cevap:" in text:
        text = text.split("Kısa cevap:")[-1].strip()

    if "Detaylı cevap:" in text:
        text = text.split("Detaylı cevap:")[0].strip()

    return text.strip()

In [26]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

llm_model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(llm_model_name)

model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("LLM loaded:", llm_model_name)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

LLM loaded: mistralai/Mistral-7B-Instruct-v0.2


In [27]:
def generate_answer(prompt, max_new_tokens=180):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Answer:" in generated_text:
        generated_text = generated_text.split("Answer:")[-1].strip()

    return generated_text

In [28]:
base_eval_questions = [
    {
        "question": "Egemenlik kime aittir?",
        "expected_answer": "Egemenlik kayıtsız şartsız Milletindir."
    },
    {
        "question": "Türkiye Cumhuriyetinin yönetim şekli nedir?",
        "expected_answer": "Türkiye Devleti bir Cumhuriyettir."
    },
    {
        "question": "Cumhurbaşkanının görev süresi kaç yıldır?",
        "expected_answer": "Cumhurbaşkanının görev süresi beş yıldır."
    },
    {
        "question": "Bir kimse en fazla kaç defa Cumhurbaşkanı seçilebilir?",
        "expected_answer": "Bir kimse en fazla iki defa Cumhurbaşkanı seçilebilir."
    }
]

base_eval_df = pd.DataFrame(base_eval_questions)
base_eval_df

,question,expected_answer
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.
2,Cumhurbaşkanının görev süresi kaç yıldır?,Cumhurbaşkanının görev süresi beş yıldır.
3,Bir kimse en fazla kaç defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...


In [29]:
fusion_small_results = []

for _, row in base_eval_df.iterrows():
    question = row["question"]
    expected_answer = row["expected_answer"]

    retrieved = hybrid_retrieve_with_reranker_fusion(
        question,
        embedding_model,
        index,
        chunks_df,
        bm25,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_CONTEXT_K,
        alpha=ALPHA,
        hybrid_weight=HYBRID_WEIGHT,
        rerank_weight=RERANK_WEIGHT
    )

    contexts = [r["chunk_text"] for r in retrieved]

    prompt = build_strict_rag_prompt(question, contexts)

    generated_answer = generate_answer(prompt)
    clean_answer = clean_generated_answer(generated_answer)

    fusion_small_results.append({
        "question": question,
        "expected_answer": expected_answer,
        "generated_answer": generated_answer,
        "clean_generated_answer": clean_answer,
        "top1_chunk_id": retrieved[0]["chunk_id"],
        "top1_source": retrieved[0]["source"],
        "top1_context": retrieved[0]["chunk_text"],
        "top1_original_rank": retrieved[0]["original_rank"],
        "top1_rerank_rank": retrieved[0]["rerank_rank"],
        "top1_fusion_score": retrieved[0]["fusion_score"]
    })

fusion_small_results_df = pd.DataFrame(fusion_small_results)
fusion_small_results_df

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_context,top1_original_rank,top1_rerank_rank,top1_fusion_score
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.,Sen Türk hukuk metinleri için çalışan dikkatli...,Egemenlik Türk Milleti'ne aittir.,chunk_000269,Türkiye Cumhuriyeti Anayasası,"Madde 6 – Egemenlik, kayıtsız şartsız Milletin...",1,1,1.000000
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.,Sen Türk hukuk metinleri için çalışan dikkatli...,"Türkiye Cumhuriyetinin yönetim şekli, Cumhurie...",chunk_000000,Türkiye Cumhuriyeti Anayasası,Türk Vatanı ve Milletinin ebedi varlığını ve Y...,1,9,0.733333
2,Cumhurbaşkanının görev süresi kaç yıldır?,Cumhurbaşkanının görev süresi beş yıldır.,Sen Türk hukuk metinleri için çalışan dikkatli...,Cumhurbaşkanının görev süresi beş yıldır.,chunk_000043,Türkiye Cumhuriyeti Anayasası,"Cumhurbaşkanı, kırk yaşını doldurmuş, yükseköğ...",1,9,0.733333
3,Bir kimse en fazla kaç defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,Sen Türk hukuk metinleri için çalışan dikkatli...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,chunk_000043,Türkiye Cumhuriyeti Anayasası,"Cumhurbaşkanı, kırk yaşını doldurmuş, yükseköğ...",1,10,0.730000


In [30]:
for i, row in fusion_small_results_df.iterrows():
    print("=" * 100)
    print("INDEX:", i)

    print("\nQUESTION:")
    print(row["question"])

    print("\nEXPECTED:")
    print(row["expected_answer"])

    print("\nCLEAN GENERATED:")
    print(row["clean_generated_answer"])

    print("\nTOP1 CHUNK ID:", row["top1_chunk_id"])
    print("TOP1 ORIGINAL RANK:", row["top1_original_rank"])
    print("TOP1 RERANK RANK:", row["top1_rerank_rank"])
    print("TOP1 FUSION SCORE:", row["top1_fusion_score"])

INDEX: 0

QUESTION:
Egemenlik kime aittir?

EXPECTED:
Egemenlik kayıtsız şartsız Milletindir.

CLEAN GENERATED:
Egemenlik Türk Milleti'ne aittir.

TOP1 CHUNK ID: chunk_000269
TOP1 ORIGINAL RANK: 1
TOP1 RERANK RANK: 1
TOP1 FUSION SCORE: 1.0
INDEX: 1

QUESTION:
Türkiye Cumhuriyetinin yönetim şekli nedir?

EXPECTED:
Türkiye Devleti bir Cumhuriyettir.

CLEAN GENERATED:
Türkiye Cumhuriyetinin yönetim şekli, Cumhuriet Sistemi (Cumhurieti Demokrasisi) olarak tanımlanmıştır.

TOP1 CHUNK ID: chunk_000000
TOP1 ORIGINAL RANK: 1
TOP1 RERANK RANK: 9
TOP1 FUSION SCORE: 0.7333333333333333
INDEX: 2

QUESTION:
Cumhurbaşkanının görev süresi kaç yıldır?

EXPECTED:
Cumhurbaşkanının görev süresi beş yıldır.

CLEAN GENERATED:
Cumhurbaşkanının görev süresi beş yıldır.

TOP1 CHUNK ID: chunk_000043
TOP1 ORIGINAL RANK: 1
TOP1 RERANK RANK: 9
TOP1 FUSION SCORE: 0.7333333333333333
INDEX: 3

QUESTION:
Bir kimse en fazla kaç defa Cumhurbaşkanı seçilebilir?

EXPECTED:
Bir kimse en fazla iki defa Cumhurbaşkanı seçileb

In [31]:
manual_scores_fusion_small = [
    1.0,
    0.0,
    1.0,
    1.0
]

fusion_small_results_df["manual_score"] = manual_scores_fusion_small

fusion_small_score = fusion_small_results_df["manual_score"].mean()

print("Fusion Reranker 4-question score:", fusion_small_score)
fusion_small_results_df

Fusion Reranker 4-question score: 0.75


,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_context,top1_original_rank,top1_rerank_rank,top1_fusion_score,manual_score
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.,Sen Türk hukuk metinleri için çalışan dikkatli...,Egemenlik Türk Milleti'ne aittir.,chunk_000269,Türkiye Cumhuriyeti Anayasası,"Madde 6 – Egemenlik, kayıtsız şartsız Milletin...",1,1,1.000000,1.0
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.,Sen Türk hukuk metinleri için çalışan dikkatli...,"Türkiye Cumhuriyetinin yönetim şekli, Cumhurie...",chunk_000000,Türkiye Cumhuriyeti Anayasası,Türk Vatanı ve Milletinin ebedi varlığını ve Y...,1,9,0.733333,0.0
2,Cumhurbaşkanının görev süresi kaç yıldır?,Cumhurbaşkanının görev süresi beş yıldır.,Sen Türk hukuk metinleri için çalışan dikkatli...,Cumhurbaşkanının görev süresi beş yıldır.,chunk_000043,Türkiye Cumhuriyeti Anayasası,"Cumhurbaşkanı, kırk yaşını doldurmuş, yükseköğ...",1,9,0.733333,1.0
3,Bir kimse en fazla kaç defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,Sen Türk hukuk metinleri için çalışan dikkatli...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,chunk_000043,Türkiye Cumhuriyeti Anayasası,"Cumhurbaşkanı, kırk yaşını doldurmuş, yükseköğ...",1,10,0.730000,1.0


In [32]:
fusion_small_results_df.to_csv(
    f"{metrics_path}/flashrank_fusion_reranker_4question_results.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    "method": "FlashRank Fusion Reranker RAG - 4 Question Sanity Evaluation",
    "manual_accuracy": fusion_small_score,
    "candidate_k": CANDIDATE_K,
    "final_context_k": FINAL_CONTEXT_K,
    "alpha": ALPHA,
    "hybrid_weight": HYBRID_WEIGHT,
    "rerank_weight": RERANK_WEIGHT
}]).to_csv(
    f"{metrics_path}/flashrank_fusion_reranker_4question_score.csv",
    index=False,
    encoding="utf-8-sig"
)

print("4-question fusion reranker results saved.")

4-question fusion reranker results saved.


In [33]:
test_eval_df = test_qa_df.sample(n=20, random_state=42).reset_index(drop=True)

print(test_eval_df.shape)
test_eval_df.head()

(20, 2)


,question,answer
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...


In [34]:
fusion_test_results = []

for _, row in tqdm(test_eval_df.iterrows(), total=len(test_eval_df)):
    question = row["question"]
    expected_answer = row["answer"]

    retrieved = hybrid_retrieve_with_reranker_fusion(
        question,
        embedding_model,
        index,
        chunks_df,
        bm25,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_CONTEXT_K,
        alpha=ALPHA,
        hybrid_weight=HYBRID_WEIGHT,
        rerank_weight=RERANK_WEIGHT
    )

    contexts = [r["chunk_text"] for r in retrieved]

    prompt = build_strict_rag_prompt(question, contexts)

    generated_answer = generate_answer(prompt)
    clean_answer = clean_generated_answer(generated_answer)

    fusion_test_results.append({
        "question": question,
        "expected_answer": expected_answer,
        "generated_answer": generated_answer,
        "clean_generated_answer": clean_answer,
        "top1_chunk_id": retrieved[0]["chunk_id"],
        "top1_source": retrieved[0]["source"],
        "top1_context": retrieved[0]["chunk_text"],
        "top1_original_rank": retrieved[0]["original_rank"],
        "top1_rerank_rank": retrieved[0]["rerank_rank"],
        "top1_fusion_score": retrieved[0]["fusion_score"],
        "retrieved_contexts": "\n\n".join(contexts)
    })

fusion_test_results_df = pd.DataFrame(fusion_test_results)
fusion_test_results_df.head()

100%|██████████| 20/20 [06:53<00:00, 20.69s/it]


,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_context,top1_original_rank,top1_rerank_rank,top1_fusion_score,retrieved_contexts
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,Sen Türk hukuk metinleri için çalışan dikkatli...,"Anayasanın 101. Maddesiyle ilgili tartışmalar,...",chunk_000210,Türkiye Cumhuriyeti Anayasası,Madde 158 – Uyuşmazlık Mahkemesi adli ve idari...,1,3,0.80,Madde 158 – Uyuşmazlık Mahkemesi adli ve idari...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...",Sen Türk hukuk metinleri için çalışan dikkatli...,"Anayasanın 10. Maddesi, Cumhurbaşkanına, Türki...",chunk_000188,Türkiye Cumhuriyeti Anayasası,"Madde 150 – Kanunların, Cumhurbaşkanlığı karar...",1,2,0.85,"Madde 150 – Kanunların, Cumhurbaşkanlığı karar..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...",Sen Türk hukuk metinleri için çalışan dikkatli...,"Hayır, Anayasanın 17. Maddesi, herkesin yaşama...",chunk_000373,Türkiye Cumhuriyeti Anayasası,"Madde 17 – Herkes, yaşama, maddi ve manevi var...",1,2,0.85,"Madde 17 – Herkes, yaşama, maddi ve manevi var..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,Sen Türk hukuk metinleri için çalışan dikkatli...,"Geçici madde 20, 1981 yılı 1 günü ve 2461 sayı...",chunk_000600,Bilgi Edinme Kanunu,Madde 20- Açıklanması veya zamanından önce açı...,1,5,0.76,Madde 20- Açıklanması veya zamanından önce açı...
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,Sen Türk hukuk metinleri için çalışan dikkatli...,"Hayır, TCK 121 ihlali sabit değildir. (Context 1)",chunk_000101,Türkiye Cumhuriyeti Anayasası,Madde 121 – (Mülga: 21/1/2017-6771/16 md.) B. ...,1,10,0.73,Madde 121 – (Mülga: 21/1/2017-6771/16 md.) B. ...


In [35]:
fusion_test_results_df.to_csv(
    f"{metrics_path}/flashrank_fusion_reranker_rag_testset_generation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Fusion reranker test generation results saved.")

Fusion reranker test generation results saved.


In [36]:
for i, row in fusion_test_results_df.iterrows():
    print("=" * 120)
    print("INDEX:", i)

    print("\nQUESTION:")
    print(row["question"])

    print("\nEXPECTED:")
    print(row["expected_answer"])

    print("\nCLEAN GENERATED:")
    print(row["clean_generated_answer"][:1000])

    print("\nTOP1 CHUNK ID:", row["top1_chunk_id"])
    print("TOP1 SOURCE:", row["top1_source"])
    print("TOP1 ORIGINAL RANK:", row["top1_original_rank"])
    print("TOP1 RERANK RANK:", row["top1_rerank_rank"])
    print("TOP1 FUSION SCORE:", row["top1_fusion_score"])

INDEX: 0

QUESTION:
Anayasanın 101. Maddesiyle İlgili Tartışmalar Nelerdir?

EXPECTED:
Cumhurbaşkanının seçilme şartlarının sınırları ve uygulanması üzerine tartışmalar olabilir.

CLEAN GENERATED:
Anayasanın 101. Maddesiyle ilgili tartışmalar, Uyuşmazlık Mahkemesi adli ve idari yargı mercileri arasındaki görev ve hüküm uyuşmazlıklarını kesin olarak çözümlemeye yetkilidir.

TOP1 CHUNK ID: chunk_000210
TOP1 SOURCE: Türkiye Cumhuriyeti Anayasası
TOP1 ORIGINAL RANK: 1
TOP1 RERANK RANK: 3
TOP1 FUSION SCORE: 0.7999999999999999
INDEX: 1

QUESTION:
Bir grup vatandaş, belirli bir etnik grubun diğerlerinden daha fazla hakka sahip olması için imza kampanyası başlatmıştır. Bu durum Anayasanın 10. Maddesi ile nasıl çelişir?

EXPECTED:
Anayasanın 10. Maddesi, herkesin kanun önünde eşit olduğunu belirtir. Bu tür bir imza kampanyası Anayasa'ya aykırıdır.

CLEAN GENERATED:
Anayasanın 10. Maddesi, Cumhurbaşkanına, Türkiye Büyük Millet Meclisinde en fazla üyeye sahip iki siyasi parti grubuna ve üye tamsa

In [37]:
fusion_test_results_df["is_valid_sample"] = True

# 03 notebook'ta index 19 invalid sample olarak çıkarılmıştı.
# Aynı evaluation setup için burada da skora dahil etmiyoruz.
fusion_test_results_df.loc[19, "is_valid_sample"] = False

manual_scores_fusion = [
    0.0,
    0.0,
    0.5,
    0.0,
    0.5,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    1.0,
    0.0,
    0.5,
    0.5,
    0.0,
    0.5,
    0.0
]

fusion_test_results_df["manual_score"] = manual_scores_fusion

valid_fusion_df = fusion_test_results_df[
    fusion_test_results_df["is_valid_sample"] == True
]

fusion_test_score = valid_fusion_df["manual_score"].mean()

print("FlashRank Fusion Reranker RAG Test Score:", fusion_test_score)
print("Valid sample count:", len(valid_fusion_df))
print("Total sample count:", len(fusion_test_results_df))

FlashRank Fusion Reranker RAG Test Score: 0.18421052631578946
Valid sample count: 19
Total sample count: 20


In [38]:
fusion_test_results_df.to_csv(
    f"{metrics_path}/flashrank_fusion_reranker_rag_testset_scored.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    "method": "FlashRank Fusion Reranker RAG - Hybrid Retrieval + ms-marco-MultiBERT-L-12 + Rank Fusion + Strict Prompt",
    "manual_accuracy": fusion_test_score,
    "valid_sample_count": len(valid_fusion_df),
    "total_sample_count": len(fusion_test_results_df),
    "candidate_k": CANDIDATE_K,
    "final_context_k": FINAL_CONTEXT_K,
    "alpha": ALPHA,
    "hybrid_weight": HYBRID_WEIGHT,
    "rerank_weight": RERANK_WEIGHT
}]).to_csv(
    f"{metrics_path}/flashrank_fusion_reranker_rag_testset_score.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Fusion reranker scored results saved.")

Fusion reranker scored results saved.


In [39]:
test_comparison_df = pd.DataFrame([
    {
        "method": "Base RAG",
        "retrieval_setup": "Hybrid retrieval top-5, context top-3",
        "prompt": "Base prompt",
        "manual_accuracy": 0.263158
    },
    {
        "method": "Strict Prompt RAG",
        "retrieval_setup": "Hybrid retrieval top-5, context top-3",
        "prompt": "Strict prompt",
        "manual_accuracy": 0.263158
    },
    {
        "method": "FlashRank Fusion Reranker RAG",
        "retrieval_setup": "Hybrid top-10 + FlashRank reranker + rank fusion top-3",
        "prompt": "Strict prompt",
        "manual_accuracy": fusion_test_score
    }
])

test_comparison_df

,method,retrieval_setup,prompt,manual_accuracy
0,Base RAG,"Hybrid retrieval top-5, context top-3",Base prompt,0.263158
1,Strict Prompt RAG,"Hybrid retrieval top-5, context top-3",Strict prompt,0.263158
2,FlashRank Fusion Reranker RAG,Hybrid top-10 + FlashRank reranker + rank fusi...,Strict prompt,0.184211


In [40]:
test_comparison_df.to_csv(
    f"{metrics_path}/rag_flashrank_fusion_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Comparison table saved.")

Comparison table saved.
